# Creating a Steganography CLI

[Steganography](https://en.wikipedia.org/wiki/Steganography) encompasses techniques for concealing a message. It differs from cryptography, which aims not to hide a message but to render it incomprehensible to anyone without the necessary key.

The method we will implement in this practical work is well-known: it involves [modifying the least significant bit of each pixel in an image](https://en.wikipedia.org/wiki/Steganography#Digital_messages). This modification is not visible to the naked eye (although it can be detected through statistical analysis).

## Work Environment

For the labs, we will use GitHub Codespaces, which provides VSCode instances within containers. The free plan of personal accounts gives 120 core hours per month, that is 60 hours on this lab's 2-core machine, and 15 GB of storage. This is more than sufficient for this practical work.

If you do not have a GitHub account, start by creating one [here](https://github.com/signup).

The labs repository is a `uv` project: the code goes in `src/steganosaure/` and the tests in `tests/`. It does not declare any dependency yet: in the Codespace's terminal, `uv add numpy scikit-image` adds those this lab needs.

Run the tests from this terminal with `uv run pytest` (`pytest` is already among the repository's development dependencies). The `tests/conftest.py` file provides two pytest fixtures, `museum_image_path` and `size_image_path`, the paths of the images in `tests/images/` that step 2 needs: a test receives them by naming them among its parameters.

[![Access the work environment](https://github.com/codespaces/badge.svg)](https://github.com/codespaces/new?hide_repo_select=true&ref=main-en&repo=1041251532&machine=basicLinux32gb&devcontainer_path=.devcontainer%2Fdevcontainer.json&location=WestEurope)

## Step 1 ⋅ Functions for Converting Strings to Bits and Vice Versa

First, we will focus on converting strings into bits (and vice versa).

A first function, `encode_string`, will take a string as an argument and return a numpy array of the corresponding bits (in utf8 encoding).

A numpy array is a homogeneous data structure (elements of the same type), which we will use in this practical work as a highly enhanced list.

For example, for the string `"ABC"`, we expect the following output array:

```python
numpy.array([0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1])
```

Indeed, `A` is encoded as `01000001` in utf8, `B` as `01000010`, and `C` as `01000011`. The expected array is simply these bits concatenated.

To code this function, here are some useful functions:
- [`str.encode`](https://docs.python.org/3/library/stdtypes.html#str.encode) which returns a `bytes` object (itself an iterable of bytes represented as integers)
- [`numpy.unpackbits`](https://numpy.org/doc/stable/reference/generated/numpy.unpackbits.html), which you can apply to a numpy array previously created from the bytes represented as integers obtained from the `bytes` object. To create the numpy array, you can use [`numpy.array`](https://numpy.org/doc/stable/reference/generated/numpy.array.html) or [`numpy.fromiter`](https://numpy.org/doc/stable/reference/generated/numpy.fromiter.html) and specify `dtype="uint8"` during the call. This argument will allow you to represent the byte values with the most suitable type.

Here is the expected function signature:

```python
def encode_string(string: str) -> numpy.ndarray:
```

The `decode_bits` function will handle the reverse transformation. The functions to use are the inverses of those needed for `encode_string`:
- [`bytes.decode`](https://docs.python.org/3/library/stdtypes.html#bytes.decode) which returns a string
- [`numpy.packbits`](https://numpy.org/doc/stable/reference/generated/numpy.packbits.html)

Here is the expected function signature:

```python
def decode_bits(bits: numpy.ndarray) -> str:
```

Instructions:

- Create the file `tests/test_string_conversion.py` and create at least one test case per function, using the given example if necessary. To compare two numpy arrays, you can use the dedicated function [`numpy.testing.assert_array_equal`](https://numpy.org/doc/stable/reference/generated/numpy.testing.assert_array_equal.html).
- Create the file `src/steganosaure/string_conversion.py` and implement these two functions inside so that the tests pass.
- Can you design a test that ensures the two functions are indeed inverses of each other? With [`hypothesis`](https://hypothesis.readthedocs.io/) (`uv add --dev hypothesis`), it can run on generated strings: `@hypothesis.given(hypothesis.strategies.text())`.

## Step 2 ⋅ Functions for Loading and Saving Images

Now we are going to code two important functions: one will load an image, and the other will save it.

**Note, for our algorithm to work, an image saved and then reloaded must be identical to the original image.**

The loading function will need to load an image into a numpy array with elements as 8-bit integers (for this, you can use the [`scikit-image`](https://scikit-image.org/) package and specifically the functions [`skimage.io.imread`](https://scikit-image.org/docs/stable/api/skimage.io.html#skimage.io.imread) and [`skimage.util.img_as_ubyte`](https://scikit-image.org/docs/stable/api/skimage.util.html#skimage.util.img_as_ubyte)). Here is the expected signature:

```python
def load_image(image_path: pathlib.Path) -> numpy.ndarray:
```

The saving function will need to save a numpy array to the specified path. Note that to ensure the reloaded image is equivalent to the saved image while maintaining good quality, it is preferable to save images in `.png` format. You can use the function [`skimage.io.imsave`](https://scikit-image.org/docs/stable/api/skimage.io.html#skimage.io.imsave). It will be useful to pass it the argument `check_contrast=False`. Here is the expected signature:

```python
def save_image(image: numpy.ndarray, image_path: pathlib.Path) -> None:
```

Instructions:

- Create the file `tests/test_image_io.py` and create at least one test case to ensure that the crucial point is respected (an image saved and then loaded is the same as the original image).
- Create the file `src/steganosaure/image_io.py` and implement these two functions inside so that the test(s) pass.

## Step 3 ⋅ Bit Manipulation Functions

We will now implement the core of the algorithm: modifying the least significant bit of each pixel to store a bit of information from the string to be hidden.

A brief reminder on bit manipulation:

```python
0b01 & 0b11 == 0b01  # bitwise AND, result bit is 1 if both bits are 1
0b01 | 0b11 == 0b11  # bitwise OR, result bit is 1 if at least one bit is 1
0b11 >> 1 == 0b01    # right shift, fills with 0s
0b11 << 1 == 0b110   # left shift, fills with 0s
```

These representations are equivalent to their decimal notation:

```python
1 & 3 == 1
1 | 3 == 3
3 >> 1 == 1
3 << 1 == 6
```

To achieve this, we will implement two functions. The first one, `retrieve_least_significant_bits`, will return a numpy array of the same size as its argument but containing only 0s or 1s: the least significant bits of the numbers contained in the argument array.

To write this function, feel free to utilize the fact that you can directly apply an operation on a numpy array (including bitwise operations):

```python
>>> 3 & 1
1
>>> 2 & 1
0
>>> numpy.array([3, 2]) & 1
array([1, 0])
```

Here is its expected signature:

```python
def retrieve_least_significant_bits(array: numpy.ndarray) -> numpy.ndarray:
```

The second function to implement, `modify_least_significant_bits`, will modify the least significant bits of the `image` so that they match the `message` array. Here is its expected signature:

```python
def modify_least_significant_bits(message: numpy.ndarray, image: numpy.ndarray) -> numpy.ndarray:
```

Instructions:

- Create the file `tests/test_bits_manipulation.py` and create at least one test case for each function.
- Create the file `src/steganosaure/bits_manipulation.py` and implement these two functions inside so that the tests pass.

## Step 4 ⋅ Functions to Manipulate numpy Arrays

The next function, `pad_array`, preprocesses the arguments of `modify_least_significant_bits` so that they are of the same size. It returns `small_array` (which will be our message to hide) padded with zeros up to the size of `big_array` (which will be the image where the message is concealed). You can use [`numpy.pad`](https://numpy.org/doc/stable/reference/generated/numpy.pad.html) for the implementation.

If `small_array` is bigger than `big_array`, the message does not fit in the image: `pad_array` must then raise a `ValueError` whose message gives both sizes.

Here is its expected signature:

```python
def pad_array(small_array: numpy.ndarray, big_array: numpy.ndarray) -> numpy.ndarray:
```

Instructions:

- Create the file `tests/test_array_reshaping.py` and write at least one test case for the zero padding, and one that checks the `ValueError` with [`pytest.raises`](https://docs.pytest.org/en/stable/how-to/assert.html#assertions-about-expected-exceptions).
- Create the file `src/steganosaure/array_reshaping.py` and implement this function inside so that the tests pass.

## Step 5 ⋅ Implementation of Steganography

We have now completed the preparatory phase and have all the necessary functions to implement steganography.

The first function of our tool, `encrypt`, will allow the user to hide a message within an image. Before applying the already developed methods, it will be necessary to use [`numpy.reshape`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) to obtain all pixel values in a one-dimensional array, then to give the modified image its original shape back. You can put these two operations in `array_reshaping.py` (`flatten_array`, `reshape_array`).

Here is its expected signature:

```python
def encrypt(message: str, input_image_path: pathlib.Path, output_image_path: pathlib.Path) -> None:
```

The second function of our tool, `decrypt`, will allow the user to retrieve the previously hidden message from an image. Here is its expected signature:

```python
def decrypt(image_path: pathlib.Path) -> str:
```

The image holds the message followed by the zeros `pad_array` added: decoded, they give null characters (`"\x00"`) at the end of the string. Remove them (`str.rstrip("\x00")`), in `decrypt` or in `decode_bits`; in the latter case, exclude this character from the strings generated in step 1: `hypothesis.strategies.text(hypothesis.strategies.characters(exclude_categories=["Cs"], exclude_characters="\x00"))`. Surrogates (category `Cs`), which `text()` excludes on its own, don't encode to utf8: a `characters` without `exclude_categories` generates them.

Instructions:

- Create the file `tests/test_steganography.py` and write at least one test case to verify the correct functioning of both functions together.
- Create the file `src/steganosaure/steganography.py` and implement these two functions inside to ensure the tests pass.

## Step 6 ⋅ The `steganosaure` Command

What remains is to turn `encrypt` and `decrypt` into a command line tool, `steganosaure`, with two subcommands:

- `encrypt`, with the arguments `message`, `input_image_path` and `output_image_path`;
- `decrypt`, with the argument `image_path`; it prints the message it finds.

Instructions:

- Create the file `src/steganosaure/cli.py`. Write in it a `create_parser` function, which builds and returns the parser with `argparse` (standard library), and a `main` function, without arguments, which uses it and calls `encrypt` or `decrypt`.
- Declare in `pyproject.toml` the `steganosaure` command, which calls the `main` function of the `steganosaure.cli` module, then run `uv sync` in the Codespace's terminal to install it.
- Create the file `tests/test_cli.py` and test the parser: `create_parser().parse_args(["decrypt", "hidden.png"])` must give the `decrypt` command and the `hidden.png` path.
- Try the command in the Codespace's terminal: `encrypt` creates the image `/tmp/hidden.png`, then `decrypt` finds the message in it.

```console
$ uv run steganosaure encrypt "Hello" tests/images/museum.jpg /tmp/hidden.png
$ uv run steganosaure decrypt /tmp/hidden.png
Hello
```

To go further, rewrite the interface with `Typer` (`uv add typer`, one function per command) and test it with `typer.testing.CliRunner`.

Useful documentation:

- [`argparse`: sub-commands](https://docs.python.org/3/library/argparse.html#subcommands)
- [Declaring a command: `[project.scripts]`](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#creating-executable-scripts)
- [`Typer`: multiple commands](https://typer.tiangolo.com/tutorial/commands/), [testing](https://typer.tiangolo.com/tutorial/testing/)

## Solution

[`solution-en` branch of the labs repository](https://github.com/shuuchuu/steganosaure/tree/solution-en). It also uses [`deal`](https://deal.readthedocs.io/), which the statement doesn't ask for: its contracts (`@deal.raises`) declare the exceptions a function may raise.